# Places365 linear probe: final paper figure

This notebook retains only the analysis needed to reproduce the **Places365 panel in Figure 2D** of the final paper:

1. evaluation loss across the 20 linear-probe epochs;
2. evaluation Macro-F1 across the 20 linear-probe epochs.

## Inputs

Four `history.json` files corresponding to the reported frozen-backbone linear probes:

- Baseline
- Fovea-Gaze
- Periph
- Periph-NF

If a history contains later exploratory fine-tuning rows, they are ignored because this notebook explicitly selects probe-stage entries only. No fine-tuning results were evaluated or reported in the published paper; strictly linear probe only!

## Output

`places365_probe_loss_and_macroF1_portrait.pdf`

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Edit these four paths if your released result files are stored elsewhere.
# Each path may point directly to history.json.

RUNS = {
    "baseline":      Path("results/places365/baseline/history.json"),
    "fovea_gaze":    Path("results/places365/fovea-gaze/history.json"),
    "periph_nonTTM": Path("results/places365/periph/history.json"),
    "periph_ttm":    Path("results/places365/periph-nf/history.json"),
}

OUT_DIR = Path("outputs")
PREFIX = "places365_probe"
OUT_PDF = OUT_DIR / f"{PREFIX}_loss_and_macroF1_portrait.pdf"

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Final plotting parameters
# ---------------------------------------------------------------------

FS_TITLE  = 16
FS_LABEL  = 14
FS_TICKS  = 12
FS_LEGEND = 12
LW_LINE   = 2.0

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": FS_TICKS,
    "axes.titlesize": FS_TITLE,
    "axes.labelsize": FS_LABEL,
    "xtick.labelsize": FS_TICKS,
    "ytick.labelsize": FS_TICKS,
    "legend.fontsize": FS_LEGEND,
})


COLOR = {
    "baseline":      "tab:blue",
    "fovea_gaze":    "tab:orange",
    "periph_nonTTM": "tab:green",
    "periph_ttm":    "tab:red",
}

# Public-facing labels from the final paper.
LABEL = {
    "baseline":      "Baseline",
    "fovea_gaze":    "Fovea-Gaze",
    "periph_nonTTM": "Periph",
    "periph_ttm":    "Periph-NF",
}

ORDER = [
    "baseline",
    "fovea_gaze",
    "periph_nonTTM",
    "periph_ttm",
]

In [ ]:
# ---------------------------------------------------------------------
# Load only linear-probe entries
# ---------------------------------------------------------------------

PROBE_STAGE_NAMES = {
    "probe",
    "linear_probe",
    "linear-probe",
    "lp",
    "linearprobe",
}

CAND = {
    "x": [
        "stage_epoch",
        "epoch",
        "global_epoch",
        "step",
    ],
    "eval_macro_f1": [
        "eval_macro_f1",
        "macro_f1",
        "eval_macroF1",
        "macroF1",
        "eval_f1",
        "macro_f1_eval",
        "val_macro_f1",
        "val_macroF1",
    ],
    "eval_loss": [
        "eval_loss",
        "loss",
        "eval_ce",
        "ce",
        "val_loss",
        "valid_loss",
        "val_ce",
    ],
}


def get_stage(entry):
    s = (
        entry.get("stage")
        or entry.get("stage_name")
        or entry.get("phase")
        or ""
    )
    return str(s).strip().lower()


def pick_key(entries, candidates):
    keys = set()

    for e in entries:
        if isinstance(e, dict):
            keys.update(e.keys())

    for k in candidates:
        if k in keys:
            return k

    # notebook's fuzzy contains-match fallback.
    lower = {
        k.lower(): k
        for k in keys
    }

    for want in candidates:
        wl = want.lower()

        for kl, orig in lower.items():
            if wl in kl:
                return orig

    return None


def extract_series(entries, y_key):
    if not entries:
        return None

    x_key = pick_key(
        entries,
        CAND["x"],
    )

    if x_key is None:
        x = np.arange(
            1,
            len(entries) + 1,
            dtype=float,
        )
    else:
        x = np.array(
            [
                e.get(x_key, np.nan)
                for e in entries
            ],
            dtype=float,
        )

    y = np.array(
        [
            e.get(y_key, np.nan)
            for e in entries
        ],
        dtype=float,
    )

    mask = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return None

    order = np.argsort(x)

    return (
        x[order],
        y[order],
    )


def load_probe_entries(history_path):
    history_path = Path(history_path)

    if not history_path.is_file():
        raise FileNotFoundError(
            f"history.json not found: {history_path}"
        )

    with history_path.open("r") as f:
        hist = json.load(f)

    if not isinstance(hist, list) or not hist:
        raise RuntimeError(
            f"Expected a non-empty list in {history_path}"
        )

    probe = [
        e
        for e in hist
        if isinstance(e, dict)
        and get_stage(e) in PROBE_STAGE_NAMES
    ]

    if not probe:
        raise RuntimeError(
            f"No linear-probe entries found in {history_path}"
        )

    return probe


def resolve_shared_metric_key(metric_name, probe_entries):
    candidates = CAND[metric_name]

    for k in candidates:
        ok = True

        for entries in probe_entries.values():
            keys = set().union(
                *[
                    e.keys()
                    for e in entries
                ]
            )

            if k not in keys:
                ok = False
                break

        if ok:
            return k

    return None


def padded_limits(
    values,
    pad_frac=0.05,
    hard_min=None,
    hard_max=None,
):
    v = np.asarray(
        values,
        dtype=float,
    )

    v = v[
        np.isfinite(v)
    ]

    if v.size == 0:
        return None

    lo = float(v.min())
    hi = float(v.max())

    if lo == hi:
        lo -= 1e-6
        hi += 1e-6

    pad = (
        hi - lo
    ) * pad_frac

    lo2 = lo - pad
    hi2 = hi + pad

    if hard_min is not None:
        lo2 = max(
            lo2,
            hard_min,
        )

    if hard_max is not None:
        hi2 = min(
            hi2,
            hard_max,
        )

    return (
        lo2,
        hi2,
    )


probe_entries = {
    name: load_probe_entries(path)
    for name, path in RUNS.items()
}

for name, entries in probe_entries.items():
    print(
        f"[OK] {name}: "
        f"probe points={len(entries)} "
        f"history.json={RUNS[name]}"
    )

In [ ]:
# ---------------------------------------------------------------------
# Build series and shared axes
# ---------------------------------------------------------------------

metrics = [
    "eval_macro_f1",
    "eval_loss",
]

shared_keys = {
    metric: resolve_shared_metric_key(
        metric,
        probe_entries,
    )
    for metric in metrics
}

series_by_metric = {
    metric: {}
    for metric in metrics
}

global_xmax = 0.0


for metric_name in metrics:
    all_y = []

    for run_name, entries in probe_entries.items():
        y_key = (
            shared_keys[metric_name]
            or pick_key(
                entries,
                CAND[metric_name],
            )
        )

        if y_key is None:
            raise RuntimeError(
                f"{run_name}: could not find metric "
                f"'{metric_name}'"
            )

        series = extract_series(
            entries,
            y_key,
        )

        if series is None:
            raise RuntimeError(
                f"{run_name}: insufficient data for "
                f"'{metric_name}'"
            )

        x, y = series

        series_by_metric[metric_name][run_name] = (
            x,
            y,
            y_key,
        )

        global_xmax = max(
            global_xmax,
            float(np.nanmax(x)),
        )

        all_y.append(y)

    flat_y = np.concatenate(
        all_y
    )

    if metric_name == "eval_macro_f1":
        series_by_metric[metric_name]["__ylims__"] = padded_limits(
            flat_y,
            pad_frac=0.03,
            hard_min=0.0,
            hard_max=1.0,
        )
    else:
        series_by_metric[metric_name]["__ylims__"] = padded_limits(
            flat_y,
            pad_frac=0.05,
        )


global_xlims = (
    1.0,
    max(
        global_xmax,
        1.0,
    ),
)

print(
    "Shared metric keys:",
    shared_keys,
)

print(
    "x limits:",
    global_xlims,
)

In [ ]:
# ---------------------------------------------------------------------
# Figure 2D: Places365 frozen-backbone linear probe
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    1,
    figsize=(6.6, 7.8),
    sharex=True,
)


def plot_from_series(
    ax,
    metric_name,
    ylabel,
    title,
    want_legend=False,
):
    ylims = (
        series_by_metric[metric_name]
        .get("__ylims__", None)
    )

    any_curve = False

    for run_name in ORDER:
        tup = (
            series_by_metric[metric_name]
            .get(run_name, None)
        )

        if tup is None:
            continue

        x, y, y_key = tup

        ax.plot(
            x,
            y,
            color=COLOR[run_name],
            linewidth=LW_LINE,
            alpha=0.90,
            label=LABEL[run_name],
        )

        any_curve = True

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )

    ax.set_xlim(
        *global_xlims
    )

    if ylims is not None:
        ax.set_ylim(
            *ylims
        )

    ax.grid(
        True,
        alpha=0.30,
    )

    if want_legend and any_curve:
        ax.legend(
            loc="upper right",
            frameon=True,
            framealpha=0.9,
            borderpad=0.3,
        )


# Top: loss with legend
plot_from_series(
    axes[0],
    metric_name="eval_loss",
    ylabel="Eval loss",
    title="Places365 linear probe: Eval loss",
    want_legend=True,
)


# Bottom: Macro-F1, no duplicate legend
plot_from_series(
    axes[1],
    metric_name="eval_macro_f1",
    ylabel="Eval macro-F1",
    title="Places365 linear probe: Eval macro-F1",
    want_legend=False,
)


axes[1].set_xlabel(
    "Probe epoch"
)

fig.tight_layout()

fig.savefig(
    OUT_PDF,
    format="pdf",
)

plt.show()

print(
    f"[OK] wrote: {OUT_PDF}"
)